# MICOM media from AGORA Table 12 and a VMH Diet Designer diet

Follows the structure of `data/inputs/vmh_high_fiber_reference.ipynb`, applied to five diets at once (stacked in one long table with a `diet` column, so every step runs once):

* `custom_diet`: your Diet Designer export (mmol/day; duplicated reactions summed)
* `agora_s12_western`, `agora_s12_high_fiber`: AGORA Supplementary Table 12 (mmol/gDW/h, from `code/01_extract_s12_table.py`)
* `agora_s12_western_bmt`, `agora_s12_high_fiber_bmt`: the same two Table 12 diets, but absorbed metabolites are multiplied by 0.8 instead of 0.1, representing a less efficient upper GI tract that lets more nutrients reach the colon

Steps: 1. floor very low values, 2. dilute metabolites absorbed in the small intestine (Recon3D exchanges), 3. add mucin cores, primary bile acids, cellulose and O2, 4. complete the medium with `complete_db_medium`, 5. validate with `check_db_medium`, 6. save each medium as a CSV (the same layout as `data/medium.csv` inside a MICOM `.qza`; no QIIME 2 artifacts are written).

Needs `micom`, `cobra`, `data/Recon3D.xml.gz` and `data/agora201_refseq216_species_1.qza`.

In [1]:
import pandas as pd
from pathlib import Path

root = Path("..")
dilutions = {"custom_diet": 0.2,
             "agora_s12_western": 0.1, "agora_s12_high_fiber": 0.1,
             "agora_s12_western_bmt": 0.8, "agora_s12_high_fiber_bmt": 0.8}

## Read the diets

The template reads `agora_metabolites.csv` for annotations. We rebuild an equivalent table from the provided AGORA media, converted to CSV by `code/02_qza_to_csv.py` (`data/media_csv/`), plus the names in Table 12. Metabolites missing from it are kept with an empty name.

In [2]:
media = [pd.read_csv(f) for f in sorted((root / "data/media_csv").glob("*.csv"))]
annotations = (pd.concat([m for m in media if "name" in m])
               .loc[:, ["metabolite", "name", "hmdb", "kegg.compound", "pubchem.compound", "inchi", "chebi"]]
               .dropna(subset=["name"]).drop_duplicates("metabolite"))
s12 = pd.read_csv(root / "data/agora_s12_table12.csv")
annotations = pd.concat([annotations, s12.loc[~s12.metabolite.isin(annotations.metabolite), ["metabolite", "name"]]])
annotations.shape

(288, 7)

In [3]:
# custom diet: sum duplicated reactions, strip IDs, map Diet Designer names to AGORA names
custom = pd.read_csv(root / "data/inputs/custom_diet_fluxes.csv").iloc[:, 0:2]
custom.columns = ["reaction", "flux"]
custom["metabolite"] = custom.reaction.str.replace("^EX_", "", regex=True).str.replace("\\[e\\]|\\(e\\)", "", regex=True)
custom["metabolite"] = custom.metabolite.replace({"oleic": "ocdcea", "palmtl": "hdcea", "tedca": "ttdca",
                                                   "xol": "chsterol", "occa": "octa", "4hpro_LT": "4hpro"})
custom = custom.groupby("metabolite", as_index=False).flux.sum()

# Table 12: one column per diet
west = s12[["metabolite", "western"]].rename(columns={"western": "flux"})
fiber = s12[["metabolite", "high_fiber"]].rename(columns={"high_fiber": "flux"})

diet = pd.concat({"custom_diet": custom,
                  "agora_s12_western": west, "agora_s12_high_fiber": fiber,
                  "agora_s12_western_bmt": west, "agora_s12_high_fiber_bmt": fiber},
                 names=["diet"]).reset_index(level=0)
diet.loc[diet.flux == 0, "flux"] = 1e-4  # bug in VMH designer where everything <1e-4 gets truncated to 0
diet

,diet,metabolite,flux
0,custom_diet,10fthf,0.001114
1,custom_diet,4hpro,1.799730
2,custom_diet,5mthf,0.001145
3,custom_diet,Lcystin,2.025749
4,custom_diet,ac,0.000100
...,...,...,...
159,agora_s12_high_fiber_bmt,thymd,1.000000
160,agora_s12_high_fiber_bmt,ura,1.000000
161,agora_s12_high_fiber_bmt,uri,1.000000
162,agora_s12_high_fiber_bmt,xan,1.000000


## Adjust for intestinal absorption

Dilute every metabolite that has an exchange in the Recon3D human model. The factor is 0.2 for the custom diet (as in the template), 0.1 for the Table 12 diets (the factor seen in `western_diet_gut_agora.qza`) and 0.8 for the two `_bmt` variants (edit `dilutions` above to change any of them). Note this reproduces the provided Western medium only roughly: Recon3D has no exchange for several ions and does have one for water.

In [4]:
from cobra.io import read_sbml_model

recon3 = read_sbml_model(root / "data/Recon3D.xml.gz")
exchanges = pd.Series([r.id for r in recon3.exchanges])
exchanges = exchanges.str.replace("__", "_").str.replace("_e$|EX_", "", regex=True)
exchanges.head()

0     5adtststerone
1    5adtststerones
2             5fthf
3             5htrp
4             5mthf
dtype: object

In [5]:
diet["dilution"] = 1.0
absorbed = diet.metabolite.isin(exchanges)
diet.loc[absorbed, "dilution"] = diet.loc[absorbed, "diet"].map(dilutions)
diet["flux"] = diet["flux"] * diet["dilution"]
diet.groupby("diet").dilution.value_counts()

diet                      dilution
agora_s12_high_fiber      0.1         109
                          1.0          55
agora_s12_high_fiber_bmt  0.8         109
                          1.0          55
agora_s12_western         0.1         109
                          1.0          55
agora_s12_western_bmt     0.8         109
                          1.0          55
custom_diet               0.2          82
                          1.0          33
Name: count, dtype: int64

## Adding host supplied components

Mucins, primary bile acids and a minuscule amount of oxygen, as in the template. Cellulose is set to 0.1 for the custom diet only (Table 12 already has its own cellulose value).

In [6]:
diet.set_index(["diet", "metabolite"], inplace=True)

for d in dilutions:
    # mucin
    for met in annotations.loc[annotations.metabolite.str.contains("core"), "metabolite"]:
        diet.loc[(d, met), "flux"] = 1

    # primary BAs
    for met in ["gchola", "tchola"]:
        diet.loc[(d, met), "flux"] = 1

    # anaerobic
    diet.loc[(d, "o2"), "flux"] = 0.001

# fiber
diet.loc[("custom_diet", "cellul"), "flux"] = 0.1

diet.reset_index(inplace=True)
diet["reaction"] = "EX_" + diet.metabolite + "(e)"
diet

,diet,metabolite,flux,dilution,reaction
0,custom_diet,10fthf,0.000223,0.2,EX_10fthf(e)
1,custom_diet,4hpro,1.799730,1.0,EX_4hpro(e)
2,custom_diet,5mthf,0.000229,0.2,EX_5mthf(e)
3,custom_diet,Lcystin,2.025749,1.0,EX_Lcystin(e)
4,custom_diet,ac,0.000020,0.2,EX_ac(e)
...,...,...,...,...,...
847,agora_s12_high_fiber_bmt,gncore2_rl,1.000000,NaN,EX_gncore2_rl(e)
848,agora_s12_high_fiber_bmt,gchola,1.000000,NaN,EX_gchola(e)
849,agora_s12_high_fiber_bmt,tchola,1.000000,NaN,EX_tchola(e)
850,agora_s12_high_fiber_bmt,o2,0.001000,NaN,EX_o2(e)


Merge with the annotations to make the table more accessible.

In [7]:
skeleton = pd.merge(diet, annotations, on="metabolite", how="left")  # left: our annotation table is incomplete

skeleton["global_id"] = skeleton.reaction
skeleton["reaction"] = "EX_" + skeleton.metabolite + "_m"
skeleton.groupby("diet").size()

diet
agora_s12_high_fiber        180
agora_s12_high_fiber_bmt    180
agora_s12_western           180
agora_s12_western_bmt       180
custom_diet                 132
dtype: int64

Save the skeleton media (before completion), so they can already be used with `code/04_compare_micom_media.py`.

In [8]:
for d, skel in skeleton.groupby("diet"):
    skel.to_csv(root / f"media_out/{d}_skeleton.csv", index=False)

## Complete the medium

Find the smallest set of additional components that lets all taxa in the model database grow.

In [ ]:
from micom.workflows.db_media import complete_db_medium

completed = {}
for d, skel in skeleton.groupby("diet"):
    manifest, imports = complete_db_medium(root / "data/agora201_refseq216_species_1.qza", skel, growth=0.05, threads=12,
                                           max_added_import=100, weights="mass")
    print(d, manifest.can_grow.value_counts().to_dict())

    filled = imports.max()
    added = filled[~filled.index.isin(skel.reaction)]
    print(f"Added flux is {added.sum():.2f}/{filled.sum():.2f} mmol/h.")
    print(added.sort_values(ascending=False)[0:20])

    added_df = added.reset_index()
    added_df.iloc[:, 0] = added_df.iloc[:, 0].str.replace("EX_|_m$", "", regex=True)
    added_df.columns = ["metabolite", "flux"]
    added_df = pd.concat([skel[["metabolite", "flux"]], added_df])

    completed[d] = pd.merge(added_df, annotations, on="metabolite", how="left")
    completed[d]["reaction"] = "EX_" + completed[d].metabolite + "_m"
    completed[d]["global_id"] = "EX_" + completed[d].metabolite + "(e)"

## Validate the media

In [ ]:
from micom.workflows.db_media import check_db_medium

for d, medium in completed.items():
    check = check_db_medium(root / "data/agora201_refseq216_species_1.qza", medium=medium, threads=12)
    print(d)
    print(check.growth_rate.describe())

## Save

In [ ]:
for d, medium in completed.items():
    medium.to_csv(root / f"media_out/{d}_micom_medium.csv", index=False)